In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("source_table_name", "", "Source Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
source_table_name = dbutils.widgets.get("source_table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f'/Volumes/adb_newcaresync/landing/landing_volume/{source_system}/{source_table_name}/{landing_timestamp}/'

# Get the list of files in the landing path (empty list if the directory doesn't exist)
try:
    files = [f for f in dbutils.fs.ls(landing_path) if not f.name.startswith('_')]
except Exception:
    files = []

# If there are no files, exit the notebook without failure
if len(files) == 0:
    print('No new files arrived')
    dbutils.notebook.exit('No new files arrived')

if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load(landing_path)    
else:
    landing_df = spark.read.format('parquet').load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_landing_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_landing_df.write.mode('append').saveAsTable(f'adb_newcaresync.bronze.{source_table_name}')